<a href="https://colab.research.google.com/github/IdrisJunaidAI/ITAI_ML_FirstRepo_IdrisJunaid/blob/main/L12_IdrisJunaid_ITAI1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 12 Lab: Ethics, Fairness, and Bias in Machine Learning

**Course:** ITAI 1371  

**Student:** Idris Junaid

**Topic**: Ethics, Fairness, and Bias in Machine Learning

**Lab:** L12

## Objective

This lab examines how machine learning models can inherit and amplify societal bias. A logistic regression model is trained on the Adult Census Income dataset and then audited across the sensitive attribute `sex`. The audit compares overall accuracy, subgroup accuracy, false positive rates, and false negative rates.

The purpose is not only to determine whether the model is accurate, but also to determine whether its errors are distributed fairly across demographic groups.

## Part 1: Understanding Algorithmic Bias

Machine learning models learn patterns from historical data. When the data contains historical discrimination, measurement problems, or unequal representation, the model can reproduce those patterns in its predictions.

Three important sources of societal bias are:

1. **Historical bias:** The data reflects past inequities, such as occupational and pay disparities.

2. **Measurement bias:** A variable or label is an imperfect proxy for the real concept of interest.

3. **Representation bias:** Some groups are underrepresented, so the model has fewer examples from which to learn.

This lab uses the Adult Census Income dataset to predict whether annual income exceeds $50,000. The dataset includes sensitive attributes such as `sex` and `race`. These attributes allow the model's outcomes and errors to be audited across groups.

In [ ]:
# Import the libraries needed for data preparation, modeling, and evaluation.
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define the column names supplied by the UCI Adult dataset documentation.
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]

# Use a local copy when available and online sources when running in Colab.
local_candidates = [
    Path("adult.data"),
    Path("/content/adult.data"),
    Path("/mnt/data/adult_original.data"),
]
online_sources = [
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "https://raw.githubusercontent.com/yanghanxy/Income_predict/refs/heads/master/Dataset/adult.data",
]
data_sources = [str(path) for path in local_candidates if path.exists()] + online_sources

df = None
last_error = None

for source in data_sources:
    try:
        candidate = pd.read_csv(
            source,
            header=None,
            names=columns,
            sep=r",\s*",
            engine="python",
            na_values="?",
        )
        if not candidate.empty:
            df = candidate
            break
    except Exception as error:
        last_error = error

if df is None:
    raise RuntimeError("The Adult dataset could not be loaded.") from last_error

# Remove records with missing values so every model input is complete.
rows_before_cleaning = len(df)
df = df.dropna().copy()
rows_after_cleaning = len(df)

# Encode the income target as 0 for <=50K and 1 for >50K.
df["income"] = df["income"].map({"<=50K": 0, ">50K": 1})

# Separate predictors from the target.
X = df.drop(columns="income")
y = df["income"]

# Preserve the class distribution in both partitions with stratification.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

# Identify numerical and categorical columns for appropriate preprocessing.
numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

# Standardize numeric variables and one hot encode categorical variables.
# The transformations are fitted only on the training data through the pipeline.
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown="ignore"), categorical_features),
)

# Train a reproducible baseline logistic regression model.
model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000, random_state=42),
)
model.fit(X_train, y_train)

overall_accuracy = model.score(X_test, y_test)

print(f"Rows before cleaning: {rows_before_cleaning:,}")
print(f"Rows after cleaning:  {rows_after_cleaning:,}")
print(f"Training rows:        {len(X_train):,}")
print(f"Testing rows:         {len(X_test):,}")
print(f"Overall accuracy:     {overall_accuracy:.2%}")

Rows before cleaning: 32,561
Rows after cleaning:  30,162
Training rows:        21,113
Testing rows:         9,049
Overall accuracy:     84.61%


## Part 2: Audit Subgroup Accuracy

High overall accuracy can hide unequal performance across demographic groups. The next function evaluates the model separately for each subgroup in the `sex` column.

Subgroup accuracy answers this question: **What percentage of predictions are correct within each group?**

In [ ]:
def get_subgroup_accuracy(
    trained_model,
    X_evaluation,
    y_evaluation,
    subgroup_column,
    subgroup_value,
):
    """Return model accuracy for one subgroup of the evaluation data."""
    subgroup_mask = X_evaluation[subgroup_column].eq(subgroup_value)
    X_subgroup = X_evaluation.loc[subgroup_mask]
    y_subgroup = y_evaluation.loc[subgroup_mask]

    if X_subgroup.empty:
        raise ValueError(f"No rows were found for subgroup: {subgroup_value}")

    return trained_model.score(X_subgroup, y_subgroup)


# Calculate accuracy separately for male and female records.
acc_male = get_subgroup_accuracy(
    model, X_test, y_test, subgroup_column="sex", subgroup_value="Male"
)
acc_female = get_subgroup_accuracy(
    model, X_test, y_test, subgroup_column="sex", subgroup_value="Female"
)

accuracy_gap = abs(acc_female - acc_male)

print(f"Accuracy for males:   {acc_male:.2%}")
print(f"Accuracy for females: {acc_female:.2%}")
print(f"Absolute accuracy gap: {accuracy_gap:.2%}")

Accuracy for males:   81.20%
Accuracy for females: 91.81%
Absolute accuracy gap: 10.61%


## Part 3: Compare False Positive and False Negative Rates

Accuracy does not identify which kinds of errors the model makes.

**False Positive Rate, FPR**

The model predicts high income for a person whose recorded income is not above $50,000.

\[
FPR = rac{FP}{FP + TN}
\]

**False Negative Rate, FNR**

The model predicts low income for a person whose recorded income is above $50,000.

\[
FNR = rac{FN}{FN + TP}
\]

Comparing these rates across groups helps reveal whether one group receives more favorable errors or more harmful errors.

In [ ]:
from sklearn.metrics import confusion_matrix


def get_error_rates(
    trained_model,
    X_evaluation,
    y_evaluation,
    subgroup_column,
    subgroup_value,
):
    """Return subgroup size, confusion matrix counts, FPR, and FNR."""
    subgroup_mask = X_evaluation[subgroup_column].eq(subgroup_value)
    X_subgroup = X_evaluation.loc[subgroup_mask]
    y_subgroup = y_evaluation.loc[subgroup_mask]

    if X_subgroup.empty:
        raise ValueError(f"No rows were found for subgroup: {subgroup_value}")

    y_pred_subgroup = trained_model.predict(X_subgroup)

    # labels=[0, 1] guarantees the expected TN, FP, FN, TP order.
    tn, fp, fn, tp = confusion_matrix(
        y_subgroup,
        y_pred_subgroup,
        labels=[0, 1],
    ).ravel()

    fpr_denominator = fp + tn
    fnr_denominator = fn + tp

    fpr = fp / fpr_denominator if fpr_denominator else float("nan")
    fnr = fn / fnr_denominator if fnr_denominator else float("nan")

    return {
        "Sample Size": len(X_subgroup),
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp,
        "False Positive Rate": fpr,
        "False Negative Rate": fnr,
    }


male_rates = get_error_rates(
    model, X_test, y_test, subgroup_column="sex", subgroup_value="Male"
)
female_rates = get_error_rates(
    model, X_test, y_test, subgroup_column="sex", subgroup_value="Female"
)

# Combine all required fairness results into one readable comparison table.
fairness_summary = pd.DataFrame(
    {
        "Male": {
            "Sample Size": male_rates["Sample Size"],
            "Accuracy": acc_male,
            "False Positive Rate": male_rates["False Positive Rate"],
            "False Negative Rate": male_rates["False Negative Rate"],
        },
        "Female": {
            "Sample Size": female_rates["Sample Size"],
            "Accuracy": acc_female,
            "False Positive Rate": female_rates["False Positive Rate"],
            "False Negative Rate": female_rates["False Negative Rate"],
        },
    }
).T

print(
    "Male confusion matrix counts: "
    f"TN={male_rates['True Negatives']}, "
    f"FP={male_rates['False Positives']}, "
    f"FN={male_rates['False Negatives']}, "
    f"TP={male_rates['True Positives']}"
)
print(
    "Female confusion matrix counts: "
    f"TN={female_rates['True Negatives']}, "
    f"FP={female_rates['False Positives']}, "
    f"FN={female_rates['False Negatives']}, "
    f"TP={female_rates['True Positives']}"
)

display(
    fairness_summary.style.format(
        {
            "Sample Size": "{:,.0f}",
            "Accuracy": "{:.2%}",
            "False Positive Rate": "{:.2%}",
            "False Negative Rate": "{:.2%}",
        }
    )
)

Male confusion matrix counts: TN=3804, FP=435, FN=720, TP=1185
Female confusion matrix counts: TN=2486, FP=72, FN=166, TP=181


,Sample Size,Accuracy,False Positive Rate,False Negative Rate
Male,"6,144",81.20%,10.26%,37.80%
Female,"2,905",91.81%,2.81%,47.84%


## Reflective Knowledge Check

### 1. Analyze the subgroup accuracy results.

The model achieved **81.20% accuracy for males** and **91.81% accuracy for females**. The female subgroup therefore had the higher overall accuracy by approximately **10.61 percentage points**. This is a meaningful difference and shows why overall accuracy alone is not an adequate fairness assessment.

However, the higher female accuracy does not automatically mean the model is fair to women. Most records in the dataset are in the lower income class, especially within the female subgroup. A model can therefore obtain high accuracy for that subgroup by correctly predicting many lower income cases while still missing a large proportion of women who truly belong to the higher income class.

### 2. Interpret the false positive and false negative errors.

The **male** false positive rate was **10.26%**, compared with **2.81%** for **females**. The model was therefore more likely to incorrectly classify a male record as having income above $50000 with the true income below this value, fifty thousand dollars.


In a loan application setting, this false positive could cause an applicant to receive a loan based on an overstated prediction of financial capacity. The applicant could be exposed to unaffordable debt, late payments, or default, while the lender could experience increased credit risk. The unequal false positive rates also mean male applicants receive substantially more favorable classification errors than female applicants.

The **female false negative rate was 47.84%**, compared with **37.80% for males**. This means the model missed nearly half of the higher income female records, which is especially important when the positive prediction provides access to an opportunity.

### 3. Decide whether the model should be approved for hiring.

I would **not approve this model for deployment in a hiring process in its current form**. In this use case, a false negative is especially harmful because a person who should qualify for consideration is screened out before receiving access to a high paying job.

The model produced a **47.84% false negative rate for females** and a **37.80% false negative rate for males**, a difference of approximately **10.04 percentage points**. Qualified women would therefore be rejected more frequently than qualified men. At the same time, the male false positive rate of **10.26%** was much higher than the female rate of **2.81%**, so men who do not meet the income based criterion would be advanced more often.

These two error patterns point in the same direction. Women are more likely to lose a deserved opportunity, while men are more likely to receive an undeserved favorable prediction. This is evidence of potential disparate impact. Before deployment, the organization should define an appropriate fairness objective, evaluate equal opportunity and equalized odds, investigate the data and proxy variables, apply a suitable mitigation method, and conduct another independent audit.

### 4. Explain whether removing `sex` would make the model fair.

Removing the `sex` column would not guarantee fairness. Other variables can act as proxies because they are correlated with gender through historical and social patterns. In this dataset, variables such as `relationship`, `marital-status`, `occupation`, `hours-per-week`, `workclass`, and possibly education related variables may allow the model to infer information associated with sex even when the explicit column is removed.

A stronger mitigation process would retain `sex` in a protected audit dataset while excluding it from ordinary prediction inputs, retrain the model, and then recompute subgroup selection rates, false positive rates, false negative rates, and true positive rates. Additional methods could include reweighting underrepresented observations, improving representation, using fairness constrained learning, and selecting decision rules that fit the legal and ethical context. Fairness must be monitored continuously because model behavior can change when the population or data distribution changes.

## Conclusion

The baseline model had a respectable overall accuracy of approximately **84.61%**, but the subgroup audit revealed substantial differences in how its errors were distributed. Female records had higher overall accuracy but also a higher false negative rate. Male records had a much higher false positive rate.

The central lesson is that aggregate performance can conceal unequal harm. Ethical machine learning therefore requires explicit fairness definitions, subgroup evaluation, context appropriate metrics, mitigation, documentation, human oversight, and continued monitoring after deployment.